In [ ]:
!pip install sentence_transformers torch ipywidgets numpy requests gensim

In [ ]:
import random
import torch 
import requests
import gensim.downloader as api
import numpy as np
from sentence_transformers import SentenceTransformer, util

In [ ]:
model = SentenceTransformer('sergeyzh/LaBSE-ru-turbo')

w2v = api.load('word2vec-ruscorpora-300')

In [ ]:
with open("top10000.txt", "r", encoding="utf-8") as f:
    all_words = [line.strip() for line in f if line.strip()]

In [ ]:
words = []
for line in all_words:
    word = line.split()[0].strip().lower()
    if len(word) > 2 and word.isalpha() and all(('а' <= c <= 'я') or c == 'ё' for c in word):
        words.append(word)


words_attempts = []
words_attempts_w2v = []

In [ ]:
clean_to_key = {}
for key in w2v.key_to_index:
    parts = key.rsplit('_', 1)
    if len(parts) == 2 and parts[1] in {'NOUN'}:
        clean = parts[0]
        if clean not in clean_to_key:
            clean_to_key[clean] = key

In [ ]:

print("words до фильтра:", len(words))
words = [w for w in words if w in clean_to_key]
print("words после фильтра:", len(words))
word_embeddings = model.encode(words, normalize_embeddings=True, show_progress_bar=True, batch_size=64).astype(np.float32)
word_embeddings_w2v = np.vstack([w2v[clean_to_key[w]] for w in words])

norms = np.linalg.norm(word_embeddings_w2v, axis=1, keepdims=True)
norms[norms == 0] = 1.0
word_embeddings_w2v /= norms

word_to_idx = {w: i for i, w in enumerate(words)} 

In [ ]:
def play_game():
    words_attempts.clear()
    words_attempts_w2v.clear()
    guessed = set()
    hinted = set()
    hinted_w2v = set()

    secret_word = random.choice(words)
    secret_idx = word_to_idx[secret_word]
    secret_embedding = word_embeddings[secret_idx]
    words_attempts.append((secret_word, 1.0, 0))

    secret_embedding_w2v = word_embeddings_w2v[secret_idx]
    words_attempts_w2v.append((secret_word, 1.0, 0))

    rating = word_embeddings @ secret_embedding
    order = np.argsort(-rating)
    rank_of_idx = np.empty_like(order)
    rank_of_idx[order] = np.arange(len(order))

    rating_w2v = word_embeddings_w2v @ secret_embedding_w2v
    order_w2v = np.argsort(-rating_w2v)
    rank_of_idx_w2v = np.empty_like(order_w2v)
    rank_of_idx_w2v[order_w2v] = np.arange(len(order_w2v))

    attempts = 0
    total = len(words)

    print(">>")

    while True:
        your_word = input("your_word: ").strip().lower()
        if not your_word:
            continue

        if your_word == "stop":
            print(f"the right word is «{secret_word}»")
            k = 20
            print("for old model:\n")
            part = np.argpartition(-rating, k)[:k]
            part = part[np.argsort(-rating[part])]
            for i in part:
                print(words[i])
            print("\nfor w2v model:")
            part = np.argpartition(-rating_w2v, k)[:k]
            part = part[np.argsort(-rating_w2v[part])]
            for i in part:
                print(words[i])
            break

        if your_word == "help":
            known_best_pos = None
            known_best_pos_w2v = None

            for (w, s, _) in words_attempts:
                if w == secret_word or w not in word_to_idx:
                    continue
                p = int(rank_of_idx[word_to_idx[w]])
                if known_best_pos is None or p < known_best_pos:
                    known_best_pos = p

            for (w, s, _) in words_attempts_w2v:
                if w == secret_word or w not in word_to_idx:
                    continue
                p = int(rank_of_idx_w2v[word_to_idx[w]])
                if known_best_pos_w2v is None or p < known_best_pos_w2v:
                    known_best_pos_w2v = p

            for w in hinted:
                if w not in word_to_idx:
                    continue
                p = int(rank_of_idx[word_to_idx[w]])
                if known_best_pos is None or p < known_best_pos:
                    known_best_pos = p

            for w in hinted_w2v:
                if w not in word_to_idx:
                    continue
                p = int(rank_of_idx_w2v[word_to_idx[w]])
                if known_best_pos_w2v is None or p < known_best_pos_w2v:
                    known_best_pos_w2v = p

           
            if known_best_pos is None:
                pos, direction = 500, -1
            elif known_best_pos <= 1:
                pos, direction = known_best_pos + 1, +1
            else:
                step = 100 if known_best_pos > 999 else 10
                pos = max(1, known_best_pos - step)
                direction = -1

            found = False
            while 0 <= pos < len(order):
                idx = int(order[pos])
                w = words[idx]
                if w != secret_word and w not in guessed and w not in hinted:
                    hinted.add(w)
                    print(f"help: try «{w}» "
                          f"(old model: sim={float(rating[idx]):.4f}, rank={pos+1}/{total})")
                    found = True
                    break
                pos += direction

            if not found:
                print("old model : help: nothing left to suggest")

            
            if known_best_pos_w2v is None:
                pos_w2v, direction_w2v = 500, -1
            elif known_best_pos_w2v <= 1:
                pos_w2v, direction_w2v = known_best_pos_w2v + 1, +1
            else:
                step_w2v = 100 if known_best_pos_w2v > 999 else 10
                pos_w2v = max(1, known_best_pos_w2v - step_w2v)
                direction_w2v = -1

            found_w2v = False
            while 0 <= pos_w2v < len(order_w2v):
                idx_w2v = int(order_w2v[pos_w2v])
                w_w2v = words[idx_w2v]
                if (w_w2v != secret_word
                        and w_w2v not in guessed
                        and w_w2v not in hinted_w2v):
                    hinted_w2v.add(w_w2v)
                    print(f"help: try «{w_w2v}» "
                          f"(w2v model: sim={float(rating_w2v[idx_w2v]):.4f}, "
                          f"rank={pos_w2v+1}/{total})")
                    found_w2v = True
                    break
                pos_w2v += direction_w2v

            if not found_w2v:
                print("w2v model : help: nothing left to suggest")
            continue

        
        attempts += 1
        guessed.add(your_word)

        if your_word == secret_word:
            print(f"you are right! {secret_word} for {attempts}")
            break

        
        your_word_embedding = model.encode([your_word], normalize_embeddings=True)[0]
        similarity = float(np.dot(your_word_embedding, secret_embedding))

        
        if your_word in clean_to_key:
            your_word_embedding_w2v = w2v[clean_to_key[your_word]].astype(np.float32)
            norm = np.linalg.norm(your_word_embedding_w2v)
            if norm > 0:
                your_word_embedding_w2v /= norm
            similarity_w2v = float(np.dot(your_word_embedding_w2v, secret_embedding_w2v))
        else:
            similarity_w2v = None

        
        if your_word in word_to_idx:
            your_word_idx = word_to_idx[your_word]
            rank = int(rank_of_idx[your_word_idx]) + 1
            print(f"old model: «{your_word}» — {rank} / {total} "
                  f"(similarity: {similarity:.3f})")
        else:
            print(f"old model: idk «{your_word}» but similarity: {similarity:.3f}")

        
        if similarity_w2v is not None:
            if your_word in word_to_idx:
                your_word_idx = word_to_idx[your_word]
                rank_w2v = int(rank_of_idx_w2v[your_word_idx]) + 1
                print(f"w2v model : «{your_word}» — {rank_w2v} / {total} "
                      f"(similarity: {similarity_w2v:.3f})")
            else:
                print(f"w2v model: idk «{your_word}» but similarity: {similarity_w2v:.3f}")
        else:
            print(f"w2v model: idk «{your_word}» (нет в словаре w2v)")

        words_attempts.append((your_word, similarity, attempts))
        words_attempts_w2v.append(
            (your_word, similarity_w2v if similarity_w2v is not None else 0.0, attempts)
        )

In [ ]:
import numpy as np

def parse_expression(expr: str):
    expr = expr.lower().strip()
    tokens = expr.split()

    terms = []
    sign = 1
    for tok in tokens:
        if tok == "+":
            sign = 1
        elif tok == "-":
            sign = -1
        else:
            w = tok.strip()
            for i in w:
                if ((i >= 'а' and i <= 'я') or (i == 'ё' or i == '-')):
                    terms.append((sign, w))
                    sign = 1
    return terms


def analogy_query(expr, model, words, word_embeddings, word_to_idx, topn=10):
    terms = parse_expression(expr)
    if not terms:
        print("wrong expr")
        return
        
    unique_words = list(dict.fromkeys(w for _, w in terms))
    enc = model.encode(unique_words, normalize_embeddings=True)
    vecs = dict(zip(unique_words, enc))

    q = np.zeros_like(next(iter(vecs.values())))
    for sign, w in terms:
        q = q + sign * vecs[w]
    n = np.linalg.norm(q)
    if n == 0:
        print("nothing")
        return
    q = q / n
    
    sims = word_embeddings @ q

    exclude = set(unique_words)
    order = np.argsort(-sims)
    result = []
    for i in order:
        w = words[i]
        if w in exclude:
            continue
        result.append((w, float(sims[i]), int(i)))
        if len(result) >= topn:
            break

            
    print(f"\ntop-{topn} close:")
    for rank, (w, s, _) in enumerate(result, 1):
        print(f"  {rank:2d}. {w:10s} sim={s:.4f}")
    print("\n")


expr = input("put expr")
analogy_query(expr, model, words, word_embeddings, word_to_idx, topn=10)

In [ ]:
import numpy as np

def parse_expression(expr: str):
    expr = expr.lower().strip()
    tokens = expr.split()

    terms = []
    sign = 1
    for tok in tokens:
        if tok == "+":
            sign = 1
        elif tok == "-":
            sign = -1
        else:
            w = tok.strip()
            for i in w:
                if ((i >= 'а' and i <= 'я') or (i == 'ё' or i == '-')):
                    terms.append((sign, w))
                    sign = 1
    return terms


def analogy_query(expr, model, w2v, clean_to_key,
                  words, word_embeddings, word_to_idx,
                  word_embeddings_w2v, topn=10):
    terms = parse_expression(expr)
    if not terms:
        print("wrong expr")
        return

    unique_words = list(dict.fromkeys(w for _, w in terms))
    exclude = set(unique_words)

    enc = model.encode(unique_words, normalize_embeddings=True)
    vecs_old = dict(zip(unique_words, enc))

    q_old = np.zeros_like(next(iter(vecs_old.values())))
    for sign, w in terms:
        q_old = q_old + sign * vecs_old[w]

    n = np.linalg.norm(q_old)
    result_old = None
    if n > 0:
        q_old = q_old / n
        sims = word_embeddings @ q_old
        order = np.argsort(-sims)
        result_old = []
        for i in order:
            w = words[i]
            if w in exclude:
                continue
            result_old.append((w, float(sims[i])))
            if len(result_old) >= topn:
                break

    # ---------- Word2Vec ----------
    vecs_w2v = {}
    missing = []
    for w in unique_words:
        key = clean_to_key.get(w)
        if key is None:
            missing.append(w)
            continue
        v = w2v[key].astype(np.float32)
        nv = np.linalg.norm(v)
        if nv > 0:
            v /= nv
        vecs_w2v[w] = v

    result_w2v = None
    if len(vecs_w2v) == len(unique_words):
        q_w2v = np.zeros_like(next(iter(vecs_w2v.values())))
        for sign, w in terms:
            q_w2v = q_w2v + sign * vecs_w2v[w]

        n = np.linalg.norm(q_w2v)
        if n > 0:
            q_w2v = q_w2v / n
            sims = word_embeddings_w2v @ q_w2v
            order = np.argsort(-sims)
            result_w2v = []
            for i in order:
                w = words[i]
                if w in exclude:
                    continue
                result_w2v.append((w, float(sims[i])))
                if len(result_w2v) >= topn:
                    break

    print(f"\n old model top {topn} close:")
    if result_old is None:
        print("  (nothing)")
    else:
        for rank, (w, s) in enumerate(result_old, 1):
            print(f"  {rank:2d}. {w:10s} sim={s:.4f}")

    print(f"\nw2v top {topn} close:")
    for rank, (w, s) in enumerate(result_w2v, 1):
        print(f"  {rank:2d}. {w:10s} sim={s:.4f}")
    print()

expr = input("put expr: ")
analogy_query(
    expr,
    model, w2v, clean_to_key,
    words, word_embeddings, word_to_idx,
    word_embeddings_w2v,
    topn=10,
)

In [ ]:
import numpy as np


def axis_top_bottom_topn(word_embeddings: np.ndarray, words: list, topn: int = 3):
    assert word_embeddings.shape[0] == len(words)
    N, D = word_embeddings.shape

    results = []
    for d in range(D):
        col = word_embeddings[:, d]

        
        idx_max = np.argpartition(-col, topn - 1)[:topn]
        idx_max = idx_max[np.argsort(-col[idx_max])]
        top_max = [(words[i], float(col[i])) for i in idx_max]

        idx_min = np.argpartition(col, topn - 1)[:topn]
        idx_min = idx_min[np.argsort(col[idx_min])]
        top_min = [(words[i], float(col[i])) for i in idx_min]

        results.append({"dim": d, "max": top_max, "min": top_min})

    return results


def _fmt_pole(pairs, width_word=5):
    return " ".join(f"{w:<{width_word}}({v:+.2f})" for w, v in pairs)


def print_axes_two_models_topn(res_old, res_w2v, max_dims: int = 50):
    n = min(len(res_labse), len(res_w2v), max_dims)

    for d in range(n):
        a = res_old[d]
        b = res_w2v[d]

        print(f"\ndim {d}")
        print(f"old model(+) : {_fmt_pole(a['max'])}")
        print(f"old model(-) : {_fmt_pole(a['min'])}")
        print(f"w2v(+) : {_fmt_pole(b['max'])}")
        print(f"w2v(-) : {_fmt_pole(b['min'])}")


In [ ]:
TOPN = 3
res_labse = axis_top_bottom_topn(word_embeddings, words, topn=TOPN)
res_w2v   = axis_top_bottom_topn(word_embeddings_w2v, words, topn=TOPN)

print_axes_two_models_topn(res_labse, res_w2v, max_dims=100)